# PathSnap — Model Experimentation, Bayesian Optimisation and HMM Route Reconstruction

One notebook covering the *Models and Training Guidelines*:

| Section | Guideline |
|---|---|
| 1–3 | Setup, upload files, leakage checks |
| 4 | Data inspection and **candidate miss rate** |
| 5 | Feature engineering (applied identically to train and test) |
| 6 | Metrics: accuracy, precision, recall, F1, macro F1, confusion matrix, log loss, **Top-1**, **route continuity**, training / inference time |
| 7 | **Seven models from seven families**, GroupKFold cross-validation on the 80 training trips, development comparison on the 10 student-test trips |
| 8 | Model comparison table and **selection by cross-validation** |
| 9 | **Bayesian hyperparameter optimisation** (Optuna TPE), tuned vs untuned |
| 10 | **HMM + Viterbi** route reconstruction, decoder tuned on training trips only |
| 11 | **Classifier-only vs classifier + HMM**: pointwise, sequence-level, continuity |
| 12 | Saved artifacts, configs, logs — downloaded as one zip |

### Files to upload (directly to Colab, not Google Drive)
1. `training_data_1.csv` — candidate table for the 80 training trips
2. `student_test_data.csv` — candidate table for the 10 student-test trips
3. `pathsnap_raw.zip` — `.arcs`, `.nodes`, `.track`, `.route` for all 90 trips (road geometry and connectivity, GPS positions and timestamps, reference routes for the route metrics)

### Leakage rules followed
- Model fitting, cross-validation, model selection, Bayesian optimisation and HMM decoder tuning use **only the 80 training trips**.
- The 10 student-test trips are used only for **development-stage comparison** (as the guideline allows), never for fitting or choosing settings.
- Cross-validation is **GroupKFold by `trip_id`**; preprocessing lives inside sklearn Pipelines, so it is fitted on training folds only.
- Features use only a GPS point's own candidates and its neighbouring points' candidates and road connectivity — **never labels or the `.route` file**.
- The evaluator-testing partition is not used anywhere.

## 1. Setup and configuration

In [ ]:
%pip install -q optuna lightgbm

In [ ]:
import os, re, json, time, glob, zipfile, warnings, platform
from collections import defaultdict
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import joblib
import optuna
import sklearn, lightgbm

from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import GroupKFold
from sklearn.metrics import (accuracy_score, precision_score, recall_score, f1_score, log_loss,
                             roc_auc_score, confusion_matrix, ConfusionMatrixDisplay)
from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import GaussianNB
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.neural_network import MLPClassifier
from lightgbm import LGBMClassifier

warnings.filterwarnings("ignore")
optuna.logging.set_verbosity(optuna.logging.WARNING)

SEED = 42
np.random.seed(SEED)

# Run profiles (free Colab has 2 CPU cores; a laptop with 8+ cores is several times faster):
#   "fast" ~15-25 min on Colab, "quick" ~45-60 min, "standard" ~1-1.5 h
PROFILE = "fast"              # "fast" (quickest), "quick", or "standard" (most thorough)
PROFILE = os.environ.get("PATHSNAP_PROFILE", PROFILE)
CFG = {
    "standard": dict(N_SPLITS=5, N_TRIALS=20, OPTUNA_FOLDS=3, OPTUNA_ROW_CAP=250_000,
                     HMM_TRIALS=30, HMM_TUNE_TRIPS=40,
                     ROW_CAP={"knn": 150_000, "random_forest": 300_000, "mlp": 300_000}),
    "quick":    dict(N_SPLITS=3, N_TRIALS=8, OPTUNA_FOLDS=2, OPTUNA_ROW_CAP=250_000,
                     HMM_TRIALS=15, HMM_TUNE_TRIPS=20,
                     ROW_CAP={"knn": 80_000, "random_forest": 150_000, "mlp": 150_000}),
    "fast":     dict(N_SPLITS=3, N_TRIALS=6, OPTUNA_FOLDS=2, OPTUNA_ROW_CAP=200_000,
                     HMM_TRIALS=12, HMM_TUNE_TRIPS=15,
                     ROW_CAP={"knn": 50_000, "random_forest": 120_000, "mlp": 100_000, "lightgbm": 400_000,
                              "decision_tree": 400_000, "logistic_regression": 400_000}),
    "smoke":    dict(N_SPLITS=3, N_TRIALS=3, OPTUNA_FOLDS=2, OPTUNA_ROW_CAP=50_000,
                     HMM_TRIALS=4, HMM_TUNE_TRIPS=6,
                     ROW_CAP={"knn": 5_000, "random_forest": 20_000, "mlp": 20_000}),
}[PROFILE]
globals().update({k: v for k, v in CFG.items()})
TARGET_TOP1 = 0.96
print("profile:", PROFILE, CFG)
print("python", platform.python_version(), "| sklearn", sklearn.__version__,
      "| lightgbm", lightgbm.__version__, "| optuna", optuna.__version__)

EXPERIMENT_LOG = []          # every experiment appends a record here (saved in section 12)
def log_event(kind, **info):
    EXPERIMENT_LOG.append({"kind": kind, "time": time.strftime("%Y-%m-%d %H:%M:%S"), **info})

## 2. Upload the three files

Run the cell; if a file is missing, a file picker opens. You can also drag the files into the
Colab **Files** panel (left sidebar) first. Files uploaded to Colab disappear when the runtime
resets — section 12 downloads all results as one zip.

In [ ]:
try:
    from google.colab import files as colab_files
    IN_COLAB = True
    DATA_DIR = "/content"
except ImportError:
    IN_COLAB = False
    DATA_DIR = os.environ.get("PATHSNAP_NB_DATA", ".")

NEEDED = {"train": ("training_data_1", ".csv", ("handoff",)),
          "test": ("student_test_data", ".csv", ("diagnostic",)),
          "raw": ("pathsnap_raw", ".zip", ())}

def find_file(keyword, ext, exclude):
    hits = [f for f in os.listdir(DATA_DIR)
            if f.lower().endswith(ext) and keyword in f.lower() and not any(x in f.lower() for x in exclude)]
    hits.sort(key=lambda f: (f.lower() != keyword + ext, len(f)))
    return os.path.join(DATA_DIR, hits[0]) if hits else None

paths = {k: find_file(*v) for k, v in NEEDED.items()}
if IN_COLAB and not all(paths.values()):
    print("Missing:", [NEEDED[k][0] + NEEDED[k][1] for k, v in paths.items() if not v], "- choose them now")
    colab_files.upload()
    paths = {k: find_file(*v) for k, v in NEEDED.items()}
missing = [NEEDED[k][0] + NEEDED[k][1] for k, v in paths.items() if not v]
assert not missing, f"Still missing: {missing}. Files in {DATA_DIR}: {sorted(os.listdir(DATA_DIR))}"
for k, v in paths.items():
    print(f"{k:5s} -> {v}  ({os.path.getsize(v)/1e6:.1f} MB)")

OUT_DIR = os.path.join(DATA_DIR, "pathsnap_results")
os.makedirs(OUT_DIR, exist_ok=True)

## 3. Load data and leakage checks

The candidate tables have one row per (GPS observation, candidate road arc). `label = 1` marks the
candidate aligned to the reference route. The raw zip gives each trip's directed road graph
(`.arcs` + `.nodes`), GPS positions and timestamps (`.track`) and reference route (`.route`).
Coordinates are projected to metres with a local flat projection centred on each trip
(trips are on different continents, so one fixed projection would distort distances).

In [ ]:
META = ["trip_id", "gps_index", "candidate_arc_id"]
TARGET = "label"
BASE_FEATURES = ["distance", "bearing_difference", "speed", "heading_change", "segment_length"]
RENAME = {"distance_m": "distance", "bearing_difference_deg": "bearing_difference",
          "speed_mps": "speed", "heading_change_deg": "heading_change", "segment_length_m": "segment_length"}

def load_table(path):
    df = pd.read_csv(path, dtype={"trip_id": str}).rename(columns=RENAME)
    miss = [c for c in META + BASE_FEATURES + [TARGET] if c not in df.columns]
    assert not miss, f"{path}: missing columns {miss}"
    df["trip_id"] = df["trip_id"].str.zfill(8)
    df[BASE_FEATURES] = df[BASE_FEATURES].apply(pd.to_numeric, errors="coerce").replace([np.inf, -np.inf], np.nan)
    df[TARGET] = df[TARGET].astype(int)
    df["gps_index"] = df["gps_index"].astype(int)
    df["candidate_arc_id"] = df["candidate_arc_id"].astype(int)
    return df.sort_values(["trip_id", "gps_index", "candidate_arc_id"]).reset_index(drop=True)

train_raw = load_table(paths["train"])
test_raw = load_table(paths["test"])

RAW_DIR = os.path.join(DATA_DIR, "pathsnap_raw")
with zipfile.ZipFile(paths["raw"]) as z:
    z.extractall(RAW_DIR)

def split_dir(name):
    hits = [d for d in glob.glob(os.path.join(RAW_DIR, "**", name), recursive=True) if os.path.isdir(d)]
    assert hits, f"no '{name}' folder inside the zip"
    return hits[0]

def load_raw_split(name):
    out = {}
    for d in sorted(glob.glob(os.path.join(split_dir(name), "*"))):
        tid = os.path.basename(d)
        arcs = np.loadtxt(os.path.join(d, f"{tid}.arcs"), dtype=np.int64, ndmin=2)
        nodes = np.loadtxt(os.path.join(d, f"{tid}.nodes"), dtype=float, ndmin=2)
        track = np.loadtxt(os.path.join(d, f"{tid}.track"), dtype=float, ndmin=2)
        route = np.loadtxt(os.path.join(d, f"{tid}.route"), dtype=np.int64, ndmin=1)
        lon0, lat0 = np.median(track[:, 0]), np.median(track[:, 1])          # per-trip local projection (metres)
        kx = 6371000.0 * np.pi / 180 * np.cos(np.radians(lat0)); ky = 6371000.0 * np.pi / 180
        P = np.c_[(track[:, 0] - lon0) * kx, (track[:, 1] - lat0) * ky]
        N = np.c_[(nodes[:, 0] - lon0) * kx, (nodes[:, 1] - lat0) * ky]
        out[tid.zfill(8)] = dict(frm=arcs[:, 0], to=arcs[:, 1], t=track[:, 2], n_gps=len(track), route=route, P=P, N=N)
    return out

RAW = {"train": load_raw_split("train"), "test": load_raw_split("student_test")}

def summary(df, name):
    pts = df.groupby(["trip_id", "gps_index"]).ngroups
    print(f"{name:13s} rows={len(df):>9,} trips={df.trip_id.nunique():>3} GPS points={pts:>8,} "
          f"positive rate={df[TARGET].mean():.3f}")
summary(train_raw, "training"); summary(test_raw, "student-test")

# ---- leakage and integrity checks
tr_trips, te_trips = set(train_raw.trip_id), set(test_raw.trip_id)
assert tr_trips.isdisjoint(te_trips), f"LEAKAGE: shared trips {sorted(tr_trips & te_trips)}"
assert tr_trips <= set(RAW["train"]), "training CSV has trips missing from raw train/"
assert te_trips <= set(RAW["test"]), "test CSV has trips missing from raw student_test/"
assert set(RAW["train"]).isdisjoint(RAW["test"]), "raw train and student_test overlap"
for name, df in [("train", train_raw), ("test", test_raw)]:
    assert not df.duplicated(META).any(), f"{name}: duplicate candidate rows"
    assert (df.groupby(["trip_id", "gps_index"])[TARGET].sum() <= 1).all(), f"{name}: >1 positive per point"
    bad = [t for t, g in df.groupby("trip_id") if g.candidate_arc_id.max() >= len(RAW[name][t]["frm"])]
    assert not bad, f"{name}: candidate_arc_id outside .arcs for trips {bad[:5]}"
print("Leakage / integrity checks passed: disjoint trips, no duplicates, <=1 positive per point, valid arc ids")
log_event("data", train_rows=len(train_raw), test_rows=len(test_raw),
          train_trips=len(tr_trips), test_trips=len(te_trips))

## 4. Data inspection and candidate miss rate

**Candidate miss rate** = share of GPS observations whose reference road segment is absent from the
candidate set (these points have no labelled positive, so no model can match them). It is an upper
bound on what any classifier or decoder can achieve.

In [ ]:
def miss_table(df, split):
    rows = []
    for tid, info in RAW[split].items():
        if tid not in set(df.trip_id):
            continue
        g = df[df.trip_id == tid]
        with_pos = g.groupby("gps_index")[TARGET].max().sum()
        rows.append(dict(trip_id=tid, gps_points=info["n_gps"], points_with_reference=int(with_pos)))
    t = pd.DataFrame(rows)
    t["candidate_miss_rate"] = 1 - t.points_with_reference / t.gps_points
    return t

miss_tr, miss_te = miss_table(train_raw, "train"), miss_table(test_raw, "test")
for name, t in [("training", miss_tr), ("student-test", miss_te)]:
    overall = 1 - t.points_with_reference.sum() / t.gps_points.sum()
    print(f"{name:13s} candidate miss rate = {overall:.2%} "
          f"(median per trip {t.candidate_miss_rate.median():.2%}, worst {t.candidate_miss_rate.max():.2%})")
MISS_RATE = {"train": 1 - miss_tr.points_with_reference.sum() / miss_tr.gps_points.sum(),
             "test": 1 - miss_te.points_with_reference.sum() / miss_te.gps_points.sum()}
print("\nWorst training trips by miss rate:")
display(miss_tr.sort_values("candidate_miss_rate", ascending=False).head(6))

fig, ax = plt.subplots(1, 3, figsize=(15, 3.5))
cpp = train_raw.groupby(["trip_id", "gps_index"]).size()
ax[0].hist(cpp, bins=range(1, cpp.max() + 2), color="#4C72B0"); ax[0].set_title("Candidates per GPS point (train)")
for lab, c in [(0, "#C44E52"), (1, "#55A868")]:
    ax[1].hist(train_raw.loc[train_raw[TARGET] == lab, "distance"], bins=50, alpha=.6, density=True, color=c,
               label="reference" if lab else "other")
ax[1].set_title("GPS-to-arc distance (m)"); ax[1].legend()
ax[2].bar(BASE_FEATURES, train_raw[BASE_FEATURES].isna().mean(), color="#8172B2")
ax[2].set_title("Missing share per feature"); ax[2].tick_params(axis="x", rotation=30)
plt.tight_layout(); plt.savefig(os.path.join(OUT_DIR, "data_inspection.png"), dpi=130); plt.show()

## 5. Feature engineering

The same function is applied to the training and student-test tables. All features are computed
from a trip's own candidate rows, its neighbouring GPS points' candidates and its road graph —
no labels and no `.route` information are used.

| Group | Features | Why |
|---|---|---|
| Base | distance, bearing_difference, speed, heading_change, segment_length | geometry and motion |
| Within-point (relative) | n_candidates, dist_rank, dist_minus_min, dist_ratio_min, dist_softmax, bear_rank, bear_minus_min, is_nearest | the model chooses *among* candidates, so how a candidate compares to its rivals matters more than raw values |
| Neighbour context | in_prev_cands, in_next_cands, is_prev_nearest, is_next_nearest, from_prev_nearest, to_next_nearest, gps_step_prev, gps_step_next | a vehicle stays on the same or a connected arc between consecutive observations |
| Direction (windowed) | along_motion_m, along_motion_norm, win_bearing_diff, window_disp_m, win_bd_rank | the 1-second GPS heading is noise at low speed; direction is measured over the smallest window of GPS points that moved ≥ 25 m and compared with the arc's direction |
| Position on the arc | proj_t, beyond_end_m, before_start_m, span_pen, span_pen_rank, signed_offset_m | near a junction several arcs are equally close; whether the point lies inside the arc's span (or already past its end) tells the incoming arc from the outgoing one |

In [ ]:
def add_features(df, raw):
    df = df.copy()
    keys = ["trip_id", "gps_index"]
    g = df.groupby(keys, sort=False)
    df["n_candidates"] = g["distance"].transform("size")
    df["dist_rank"] = g["distance"].rank(method="first")
    dmin = g["distance"].transform("min")
    df["dist_minus_min"] = df["distance"] - dmin
    df["dist_ratio_min"] = (df["distance"] + 1.0) / (dmin + 1.0)
    w = np.exp(-df["dist_minus_min"] / 10.0)
    df["dist_softmax"] = w / w.groupby([df.trip_id, df.gps_index]).transform("sum")
    df["bear_rank"] = g["bearing_difference"].rank(method="min")
    df["bear_minus_min"] = df["bearing_difference"] - g["bearing_difference"].transform("min")
    df["is_nearest"] = (df["dist_rank"] == 1).astype(int)

    # neighbouring observed GPS points (points without candidates are skipped)
    pts = df[keys].drop_duplicates().sort_values(keys)
    pts["prev_gps"] = pts.groupby("trip_id")["gps_index"].shift(1)
    pts["next_gps"] = pts.groupby("trip_id")["gps_index"].shift(-1)
    nearest = df.loc[df.is_nearest == 1, keys + ["candidate_arc_id"]].rename(columns={"candidate_arc_id": "nearest_arc"})
    pts = pts.merge(nearest.rename(columns={"gps_index": "prev_gps", "nearest_arc": "prev_nearest"}), on=["trip_id", "prev_gps"], how="left")
    pts = pts.merge(nearest.rename(columns={"gps_index": "next_gps", "nearest_arc": "next_nearest"}), on=["trip_id", "next_gps"], how="left")
    df = df.merge(pts, on=keys, how="left")

    cand = df[keys + ["candidate_arc_id"]].assign(_hit=1)
    df = df.merge(cand.rename(columns={"gps_index": "prev_gps", "_hit": "in_prev_cands"}),
                  on=["trip_id", "prev_gps", "candidate_arc_id"], how="left")
    df = df.merge(cand.rename(columns={"gps_index": "next_gps", "_hit": "in_next_cands"}),
                  on=["trip_id", "next_gps", "candidate_arc_id"], how="left")
    df[["in_prev_cands", "in_next_cands"]] = df[["in_prev_cands", "in_next_cands"]].fillna(0)
    df["is_prev_nearest"] = (df.candidate_arc_id == df.prev_nearest).astype(int)
    df["is_next_nearest"] = (df.candidate_arc_id == df.next_nearest).astype(int)
    df["gps_step_prev"] = df.gps_index - df.prev_gps
    df["gps_step_next"] = df.next_gps - df.gps_index

    # road connectivity: arc follows the previous nearest arc / leads into the next nearest arc
    from_prev = np.zeros(len(df), dtype=int)
    to_next = np.zeros(len(df), dtype=int)
    for tid, idx in df.groupby("trip_id").indices.items():
        frm, to = raw[tid]["frm"], raw[tid]["to"]
        a = df.candidate_arc_id.values[idx]
        pn, nn = df.prev_nearest.values[idx], df.next_nearest.values[idx]
        ok_p, ok_n = ~np.isnan(pn), ~np.isnan(nn)
        from_prev[idx[ok_p]] = ((to[pn[ok_p].astype(int)] == frm[a[ok_p]]) | (pn[ok_p].astype(int) == a[ok_p])).astype(int)
        to_next[idx[ok_n]] = ((to[a[ok_n]] == frm[nn[ok_n].astype(int)]) | (nn[ok_n].astype(int) == a[ok_n])).astype(int)
    df["from_prev_nearest"], df["to_next_nearest"] = from_prev, to_next
    df = df.drop(columns=["prev_gps", "next_gps", "prev_nearest", "next_nearest"])
    return df.sort_values(["trip_id", "gps_index", "candidate_arc_id"]).reset_index(drop=True)

def window_vectors(P, min_disp=25.0, kmax=20):
    # displacement over the smallest symmetric window (i-k .. i+k) that moved >= min_disp metres
    n = len(P); V = np.zeros((n, 2)); done = np.zeros(n, bool); idx = np.arange(n)
    for k in range(1, kmax + 1):
        a, b = np.clip(idx - k, 0, n - 1), np.clip(idx + k, 0, n - 1); v = P[b] - P[a]
        ok = (~done) & (np.hypot(v[:, 0], v[:, 1]) >= min_disp); V[ok] = v[ok]; done |= ok
    a, b = np.clip(idx - kmax, 0, n - 1), np.clip(idx + kmax, 0, n - 1); V[~done] = (P[b] - P[a])[~done]
    return V

def add_geometry_features(df, raw):
    cols = ["along_motion_m", "win_bearing_diff", "window_disp_m", "proj_t", "beyond_end_m", "before_start_m",
            "signed_offset_m", "pos_m"]
    out = {c: np.full(len(df), np.nan) for c in cols}
    for tid, idx in df.groupby("trip_id").indices.items():
        r = raw[tid]; V = window_vectors(r["P"])
        gi = df.gps_index.values[idx]; a = df.candidate_arc_id.values[idx]
        A, B = r["N"][r["frm"][a]], r["N"][r["to"][a]]; av = B - A
        L = np.maximum(np.hypot(av[:, 0], av[:, 1]), 1e-6); u = av / L[:, None]
        v = V[gi]; vn = np.hypot(v[:, 0], v[:, 1]); p = r["P"][gi]
        out["along_motion_m"][idx] = (v * u).sum(1)
        out["window_disp_m"][idx] = vn
        out["win_bearing_diff"][idx] = np.degrees(np.arccos(np.clip((v * u).sum(1) / np.maximum(vn, 1e-6), -1, 1)))
        t = ((p - A) * u).sum(1) / L
        out["proj_t"][idx] = t
        out["pos_m"][idx] = t * L                                   # position along the arc (used by the HMM)
        out["beyond_end_m"][idx] = np.maximum(0, (t - 1) * L)
        out["before_start_m"][idx] = np.maximum(0, -t * L)
        out["signed_offset_m"][idx] = u[:, 0] * (p - A)[:, 1] - u[:, 1] * (p - A)[:, 0]
    df = df.copy()
    for c in cols:
        df[c] = out[c]
    df["along_motion_norm"] = df.along_motion_m / np.maximum(df.window_disp_m, 1.0)
    df["span_pen"] = df.beyond_end_m + df.before_start_m
    g = df.groupby(["trip_id", "gps_index"], sort=False)
    df["win_bd_rank"] = g["win_bearing_diff"].rank(method="min")
    df["span_pen_rank"] = g["span_pen"].rank(method="min")
    return df

t0 = time.perf_counter()
train = add_geometry_features(add_features(train_raw, RAW["train"]), RAW["train"])
test = add_geometry_features(add_features(test_raw, RAW["test"]), RAW["test"])
FEATURES = BASE_FEATURES + ["n_candidates", "dist_rank", "dist_minus_min", "dist_ratio_min", "dist_softmax",
                            "bear_rank", "bear_minus_min", "is_nearest", "in_prev_cands", "in_next_cands",
                            "is_prev_nearest", "is_next_nearest", "from_prev_nearest", "to_next_nearest",
                            "gps_step_prev", "gps_step_next",
                            "along_motion_m", "along_motion_norm", "win_bearing_diff", "window_disp_m", "win_bd_rank",
                            "proj_t", "beyond_end_m", "before_start_m", "span_pen", "span_pen_rank", "signed_offset_m"]
KNN_FEATURES = ["distance", "dist_minus_min", "along_motion_norm", "win_bearing_diff", "span_pen",
                "is_prev_nearest", "is_next_nearest", "from_prev_nearest", "to_next_nearest"]
assert len(train) == len(train_raw) and len(test) == len(test_raw), "feature merge changed the row count"
assert not set(FEATURES) & set(META + [TARGET]), "ID or label used as a feature"
assert "pos_m" not in FEATURES  # absolute position is only used by the HMM
print(f"{len(FEATURES)} features built in {time.perf_counter()-t0:.1f}s")
display(train[FEATURES].describe().T[["mean", "std", "min", "max"]].round(3))

## 6. Metrics

- **Observation-level Top-1 accuracy (primary metric)** — for each GPS point, the highest-scoring
  candidate is compared with the reference arc. This is exactly the map-matching decision, so it is
  used to select models; log loss is the tie-breaker because the HMM consumes the probabilities.
- Row-level accuracy / precision / recall / F1 / macro F1 / confusion matrix at threshold 0.5.
- **Route continuity** — share of consecutive predicted-arc changes that are physically connected
  (`to_node(previous) == from_node(next)`) in the directed road graph.

In [ ]:
POINT = ["trip_id", "gps_index"]

def top1_choice(df, score):
    d = df[POINT + ["candidate_arc_id", TARGET]].copy()
    d["score"] = np.asarray(score)
    idx = d.groupby(POINT, sort=False)["score"].idxmax()
    return d.loc[idx].sort_values(POINT).reset_index(drop=True)

def route_continuity(choice, split):
    ok = total = 0
    for tid, g in choice.groupby("trip_id"):
        a = g.sort_values("gps_index")["candidate_arc_id"].to_numpy()
        frm, to = RAW[split][tid]["frm"], RAW[split][tid]["to"]
        ch = a[1:] != a[:-1]
        ok += int((to[a[:-1][ch]] == frm[a[1:][ch]]).sum()); total += int(ch.sum())
    return ok / total if total else 1.0

def compute_metrics(df, proba, split=None):
    y = df[TARGET].to_numpy(); p = np.clip(np.asarray(proba), 1e-7, 1 - 1e-7); yhat = (p >= 0.5).astype(int)
    ch = top1_choice(df, p)
    m = {"accuracy": accuracy_score(y, yhat), "precision": precision_score(y, yhat, zero_division=0),
         "recall": recall_score(y, yhat, zero_division=0), "f1": f1_score(y, yhat, zero_division=0),
         "macro_f1": f1_score(y, yhat, average="macro", zero_division=0),
         "log_loss": log_loss(y, p, labels=[0, 1]), "roc_auc": roc_auc_score(y, p) if len(set(y)) > 1 else np.nan,
         "top1_accuracy": ch[TARGET].mean()}
    if split is not None:
        m["route_continuity"] = route_continuity(ch, split)
    return m

# Non-ML reference point: always pick the nearest candidate
for name, df, split in [("training", train, "train"), ("student-test", test, "test")]:
    ch = top1_choice(df, -df["distance"].to_numpy())
    print(f"Nearest-candidate baseline ({name}): Top-1 = {ch[TARGET].mean():.4f}, "
          f"route continuity = {route_continuity(ch, split):.4f}")
BASELINE_TEST = top1_choice(test, -test["distance"].to_numpy())[TARGET].mean()

## 7. Train and evaluate seven models (seven algorithm families)

Each model: GroupKFold cross-validation on the 80 training trips (out-of-fold probabilities kept),
then one fit on all training trips and one evaluation on the 10 student-test trips.
Slow models (KNN, Random Forest, MLP) fit on a random sample of each fold's **training** rows
(`ROW_CAP`), recorded in the results — validation and test rows are never sampled away.

In [ ]:
def imp():
    return SimpleImputer(strategy="median", add_indicator=True)

MODELS = {
    "logistic_regression": dict(family="Linear", features=FEATURES,
        make=lambda **p: Pipeline([("imp", imp()), ("sc", StandardScaler()),
                                   ("clf", LogisticRegression(max_iter=2000, C=p.get("C", 1.0), random_state=SEED))]),
        config="C=1.0, L2, lbfgs, standardised"),
    "gaussian_nb": dict(family="Probabilistic", features=FEATURES,
        make=lambda **p: Pipeline([("imp", imp()), ("clf", GaussianNB(var_smoothing=p.get("var_smoothing", 1e-9)))]),
        config="var_smoothing=1e-9"),
    "knn": dict(family="Instance-based", features=KNN_FEATURES,
        make=lambda **p: Pipeline([("imp", imp()), ("sc", StandardScaler()),
                                   ("clf", KNeighborsClassifier(n_neighbors=p.get("n_neighbors", 25), weights="distance", n_jobs=-1))]),
        config="k=25, distance-weighted, 9 standardised features"),
    "decision_tree": dict(family="Tree", features=FEATURES,
        make=lambda **p: Pipeline([("imp", imp()), ("clf", DecisionTreeClassifier(
            max_depth=p.get("max_depth", 12), min_samples_leaf=p.get("min_samples_leaf", 50), random_state=SEED))]),
        config="max_depth=12, min_samples_leaf=50"),
    "random_forest": dict(family="Ensemble (bagging)", features=FEATURES,
        make=lambda **p: Pipeline([("imp", imp()), ("clf", RandomForestClassifier(
            n_estimators=p.get("n_estimators", 150), max_depth=p.get("max_depth", 18),
            min_samples_leaf=p.get("min_samples_leaf", 5), max_features=p.get("max_features", "sqrt"),
            n_jobs=-1, random_state=SEED))]),
        config="150 trees, max_depth=18, min_samples_leaf=5"),
    "lightgbm": dict(family="Boosting", features=FEATURES,
        make=lambda **p: Pipeline([("imp", imp()), ("clf", LGBMClassifier(
            n_estimators=p.get("n_estimators", 400), learning_rate=p.get("learning_rate", 0.05),
            num_leaves=p.get("num_leaves", 63), min_child_samples=p.get("min_child_samples", 40),
            subsample=p.get("subsample", 0.8), subsample_freq=1, colsample_bytree=p.get("colsample_bytree", 0.8),
            reg_lambda=p.get("reg_lambda", 1.0), random_state=SEED, n_jobs=-1, verbose=-1))]),
        config="400 trees, lr=0.05, 63 leaves"),
    "mlp": dict(family="Neural network", features=FEATURES,
        make=lambda **p: Pipeline([("imp", imp()), ("sc", StandardScaler()), ("clf", MLPClassifier(
            hidden_layer_sizes=p.get("hidden_layer_sizes", (64, 32)), alpha=p.get("alpha", 1e-4),
            learning_rate_init=p.get("learning_rate_init", 1e-3), early_stopping=True, max_iter=60,
            random_state=SEED))]),
        config="hidden (64, 32), early stopping, standardised"),
}

def sample_rows(idx, cap, seed):
    if cap is None or len(idx) <= cap:
        return idx
    return np.sort(np.random.default_rng(seed).choice(idx, cap, replace=False))

def run_cv(name, params=None, n_splits=N_SPLITS, row_cap="default", data=None, quiet=False):
    spec, params = MODELS[name], params or {}
    data = train if data is None else data
    cap = ROW_CAP.get(name) if row_cap == "default" else row_cap
    X, y, groups = data[spec["features"]], data[TARGET], data["trip_id"]
    oof = np.full(len(data), np.nan); rows = []
    for fold, (tr, va) in enumerate(GroupKFold(n_splits=n_splits).split(X, y, groups)):
        assert set(groups.iloc[tr]).isdisjoint(groups.iloc[va]), "trip leakage between folds"
        fit_idx = sample_rows(tr, cap, SEED + fold)
        model = spec["make"](**params)
        t0 = time.perf_counter(); model.fit(X.iloc[fit_idx], y.iloc[fit_idx]); fit_s = time.perf_counter() - t0
        t0 = time.perf_counter(); p = model.predict_proba(X.iloc[va])[:, 1]; pred_s = time.perf_counter() - t0
        oof[va] = p
        m = compute_metrics(data.iloc[va], p); m.update(fold=fold, fit_s=fit_s, predict_s=pred_s, fit_rows=len(fit_idx))
        rows.append(m)
        if not quiet:
            print(f"  fold {fold}: top1={m['top1_accuracy']:.4f} macroF1={m['macro_f1']:.4f} "
                  f"logloss={m['log_loss']:.4f} (fit {fit_s:.1f}s on {len(fit_idx):,} rows)", flush=True)
    return pd.DataFrame(rows), oof

def fit_and_test(name, params=None):
    spec, params = MODELS[name], params or {}
    idx = sample_rows(np.arange(len(train)), ROW_CAP.get(name), SEED)
    model = spec["make"](**params)
    t0 = time.perf_counter(); model.fit(train[spec["features"]].iloc[idx], train[TARGET].iloc[idx]); fit_s = time.perf_counter() - t0
    t0 = time.perf_counter(); p = model.predict_proba(test[spec["features"]])[:, 1]; pred_s = time.perf_counter() - t0
    m = compute_metrics(test, p, split="test")
    m.update(train_fit_s=fit_s, inference_s=pred_s, inference_ms_per_1k_rows=1000 * pred_s / len(test) * 1000)
    return model, p, m

In [ ]:
RESULTS, CV_FOLDS, OOF, TEST_PROBA, FITTED = [], [], {}, {}, {}
for name, spec in MODELS.items():
    print(f"\n=== {name} ({spec['family']}) — {spec['config']}", flush=True)
    t_all = time.perf_counter()
    cv, oof = run_cv(name)
    model, p_test, tm = fit_and_test(name)
    cv["model"] = name; CV_FOLDS.append(cv); OOF[name] = oof; TEST_PROBA[name] = p_test; FITTED[name] = model
    row = {"model": name, "family": spec["family"], "config": spec["config"],
           "n_features": len(spec["features"]), "row_cap": ROW_CAP.get(name)}
    for k in ["top1_accuracy", "macro_f1", "f1", "precision", "recall", "accuracy", "log_loss", "roc_auc"]:
        row[f"cv_{k}"] = cv[k].mean(); row[f"cv_{k}_std"] = cv[k].std()
    row["cv_fit_s_mean"] = cv["fit_s"].mean()
    row["cv_route_continuity"] = route_continuity(top1_choice(train, oof), "train")
    for k, v in tm.items():
        row[f"test_{k}"] = v
    row["total_s"] = time.perf_counter() - t_all
    RESULTS.append(row)
    log_event("model", **{k: v for k, v in row.items() if not isinstance(v, float) or np.isfinite(v)})
    print(f"  -> CV top1 {row['cv_top1_accuracy']:.4f} ± {row['cv_top1_accuracy_std']:.4f} | "
          f"student-test top1 {tm['top1_accuracy']:.4f} | continuity {tm['route_continuity']:.4f} | {row['total_s']:.0f}s")

comparison = pd.DataFrame(RESULTS)
pd.concat(CV_FOLDS).to_csv(os.path.join(OUT_DIR, "cv_folds_all_models.csv"), index=False)
comparison.to_csv(os.path.join(OUT_DIR, "model_comparison.csv"), index=False)

## 8. Model comparison and selection

**Selection rule (decided before looking at student-test results):** highest mean cross-validated
Top-1 accuracy on the 80 training trips; if two models are within one fold-standard-deviation, the
one with lower CV log loss wins (better probabilities for the HMM). Student-test columns are shown
for development comparison only.

In [ ]:
show = ["model", "family", "config", "cv_top1_accuracy", "cv_top1_accuracy_std", "cv_macro_f1", "cv_log_loss",
        "test_top1_accuracy", "test_precision", "test_recall", "test_f1", "test_macro_f1", "test_log_loss",
        "test_route_continuity", "cv_fit_s_mean", "test_inference_s"]
table = comparison[show].sort_values("cv_top1_accuracy", ascending=False).reset_index(drop=True)
display(table.round(4))

best = table.iloc[0]
close = table[table.cv_top1_accuracy >= best.cv_top1_accuracy - best.cv_top1_accuracy_std]
SELECTED = close.sort_values("cv_log_loss").iloc[0]["model"]
print(f"Selected for tuning and the HMM: {SELECTED}")
print("Reason: best (or statistically tied best) CV Top-1 on the training trips; among ties, lowest CV log loss.")
log_event("selection", selected=SELECTED, candidates_within_1std=list(close.model))

fig, ax = plt.subplots(1, 3, figsize=(16, 4))
t2 = table.sort_values("cv_top1_accuracy")
ax[0].barh(t2.model, t2.cv_top1_accuracy, xerr=t2.cv_top1_accuracy_std, color="#4C72B0"); ax[0].set_title("CV Top-1 (mean ± std)")
ax[0].set_xlim(max(0, t2.cv_top1_accuracy.min() - 0.05), 1)
ax[1].barh(t2.model, t2.test_top1_accuracy, color="#55A868"); ax[1].set_title("Student-test Top-1")
ax[1].set_xlim(max(0, t2.test_top1_accuracy.min() - 0.05), 1)
ax[2].barh(t2.model, t2.cv_log_loss, color="#C44E52"); ax[2].set_title("CV log loss (lower is better)")
plt.tight_layout(); plt.savefig(os.path.join(OUT_DIR, "model_comparison.png"), dpi=130); plt.show()

fig, ax = plt.subplots(figsize=(5, 4))
yhat = (TEST_PROBA[SELECTED] >= 0.5).astype(int)
ConfusionMatrixDisplay(confusion_matrix(test[TARGET], yhat), display_labels=["other arc", "reference arc"]).plot(ax=ax, colorbar=False)
ax.set_title(f"{SELECTED}: student-test confusion matrix"); plt.tight_layout()
plt.savefig(os.path.join(OUT_DIR, f"confusion_{SELECTED}.png"), dpi=130); plt.show()

## 9. Bayesian hyperparameter optimisation (Optuna TPE)

- **Model:** the selected model from section 8.
- **Objective:** maximise mean observation-level Top-1 accuracy under GroupKFold
  (`OPTUNA_FOLDS` folds) on the 80 training trips only.
- **Search space:** justified per model (capacity vs. regularisation); listed in the cell.
- The tuned configuration is refitted on all training trips, evaluated on the student-test trips and
  compared with the untuned model. The final model is the one with better **CV** Top-1 (not test).

In [ ]:
SPACES = {
    "lightgbm": lambda t: dict(
        n_estimators=t.suggest_int("n_estimators", 200, 900, step=100),      # more trees with lower lr
        learning_rate=t.suggest_float("learning_rate", 0.02, 0.2, log=True),
        num_leaves=t.suggest_int("num_leaves", 15, 255, log=True),            # tree complexity
        min_child_samples=t.suggest_int("min_child_samples", 10, 300, log=True),  # leaf-size regularisation
        subsample=t.suggest_float("subsample", 0.6, 1.0),
        colsample_bytree=t.suggest_float("colsample_bytree", 0.5, 1.0),
        reg_lambda=t.suggest_float("reg_lambda", 1e-3, 10.0, log=True)),
    "random_forest": lambda t: dict(
        n_estimators=t.suggest_int("n_estimators", 100, 300, step=50),
        max_depth=t.suggest_int("max_depth", 8, 30),
        min_samples_leaf=t.suggest_int("min_samples_leaf", 1, 50, log=True),
        max_features=t.suggest_categorical("max_features", ["sqrt", "log2", 0.5])),
    "decision_tree": lambda t: dict(max_depth=t.suggest_int("max_depth", 4, 30),
                                    min_samples_leaf=t.suggest_int("min_samples_leaf", 1, 500, log=True)),
    "logistic_regression": lambda t: dict(C=t.suggest_float("C", 1e-3, 100, log=True)),
    "knn": lambda t: dict(n_neighbors=t.suggest_int("n_neighbors", 5, 101, step=2)),
    "gaussian_nb": lambda t: dict(var_smoothing=t.suggest_float("var_smoothing", 1e-12, 1e-2, log=True)),
    "mlp": lambda t: dict(hidden_layer_sizes=tuple(int(x) for x in t.suggest_categorical("hidden_layers", ["32", "64-32", "128-64", "128-64-32"]).split("-")),
                          alpha=t.suggest_float("alpha", 1e-6, 1e-2, log=True),
                          learning_rate_init=t.suggest_float("learning_rate_init", 1e-4, 1e-2, log=True)),
}

def objective(trial):
    params = SPACES[SELECTED](trial)
    cap = min(OPTUNA_ROW_CAP, ROW_CAP.get(SELECTED) or OPTUNA_ROW_CAP)
    cv, _ = run_cv(SELECTED, params, n_splits=OPTUNA_FOLDS, row_cap=cap, quiet=True)
    trial.set_user_attr("cv_log_loss", float(cv.log_loss.mean()))
    trial.set_user_attr("fit_s", float(cv.fit_s.sum()))
    return float(cv.top1_accuracy.mean())

study = optuna.create_study(direction="maximize", sampler=optuna.samplers.TPESampler(seed=SEED),
                            study_name=f"pathsnap_{SELECTED}")
t0 = time.perf_counter()
study.optimize(objective, n_trials=N_TRIALS, show_progress_bar=False,
               callbacks=[lambda s, t: print(f"  trial {t.number:2d}: top1={t.value:.4f} params={t.params}", flush=True)])
TUNING_S = time.perf_counter() - t0
trials = study.trials_dataframe()
trials.to_csv(os.path.join(OUT_DIR, "optuna_trials.csv"), index=False)
BEST_PARAMS = SPACES[SELECTED](optuna.trial.FixedTrial(study.best_params))   # decoded (e.g. MLP layer sizes)
print(f"\nBest CV Top-1 during search: {study.best_value:.4f} with {BEST_PARAMS}")
print(f"Optimisation cost: {N_TRIALS} trials x {OPTUNA_FOLDS} folds in {TUNING_S/60:.1f} min")
log_event("optuna", model=SELECTED, n_trials=N_TRIALS, folds=OPTUNA_FOLDS, best_value=study.best_value,
          best_params={k: str(v) for k, v in BEST_PARAMS.items()}, seconds=TUNING_S)

plt.figure(figsize=(7, 3.5))
plt.plot(trials.number, trials.value, "o", alpha=.6, label="trial"); plt.plot(trials.number, trials.value.cummax(), "-", label="best so far")
plt.xlabel("trial"); plt.ylabel("CV Top-1"); plt.title(f"Optuna optimisation history ({SELECTED})"); plt.legend()
plt.tight_layout(); plt.savefig(os.path.join(OUT_DIR, "optuna_history.png"), dpi=130); plt.show()

In [ ]:
print("Tuned model: full GroupKFold CV + student-test evaluation")
cv_tuned, oof_tuned = run_cv(SELECTED, BEST_PARAMS)
model_tuned, p_test_tuned, tm_tuned = fit_and_test(SELECTED, BEST_PARAMS)
untuned = comparison.set_index("model").loc[SELECTED]
tuned_vs = pd.DataFrame({
    "untuned": {"cv_top1": untuned.cv_top1_accuracy, "cv_log_loss": untuned.cv_log_loss,
                "test_top1": untuned.test_top1_accuracy, "test_macro_f1": untuned.test_macro_f1,
                "test_log_loss": untuned.test_log_loss, "test_route_continuity": untuned.test_route_continuity},
    "tuned": {"cv_top1": cv_tuned.top1_accuracy.mean(), "cv_log_loss": cv_tuned.log_loss.mean(),
              "test_top1": tm_tuned["top1_accuracy"], "test_macro_f1": tm_tuned["macro_f1"],
              "test_log_loss": tm_tuned["log_loss"], "test_route_continuity": tm_tuned["route_continuity"]}})
display(tuned_vs.round(4))
tuned_vs.to_csv(os.path.join(OUT_DIR, "tuned_vs_untuned.csv"))

USE_TUNED = cv_tuned.top1_accuracy.mean() > untuned.cv_top1_accuracy      # decided on CV, not test; ties keep the simpler default
FINAL_PARAMS = BEST_PARAMS if USE_TUNED else {}
FINAL_MODEL = model_tuned if USE_TUNED else FITTED[SELECTED]
FINAL_OOF = oof_tuned if USE_TUNED else OOF[SELECTED]
FINAL_TEST_P = p_test_tuned if USE_TUNED else TEST_PROBA[SELECTED]
print(f"Final classifier: {SELECTED} ({'tuned' if USE_TUNED else 'untuned'} — chosen on CV Top-1)")
joblib.dump(FINAL_MODEL, os.path.join(OUT_DIR, "final_model.joblib"))
log_event("final_model", model=SELECTED, tuned=bool(USE_TUNED))

clf = FINAL_MODEL.named_steps["clf"]
if hasattr(clf, "feature_importances_"):
    names = FINAL_MODEL.named_steps["imp"].get_feature_names_out(MODELS[SELECTED]["features"])
    imp_s = pd.Series(clf.feature_importances_, index=names).sort_values()
    imp_s.tail(20).plot.barh(figsize=(6, 6), color="#4C72B0", title=f"{SELECTED} feature importance")
    plt.tight_layout(); plt.savefig(os.path.join(OUT_DIR, "feature_importance.png"), dpi=130); plt.show()

## 10. HMM and Viterbi route reconstruction

**Hidden states:** at GPS observation *t*, the states are that observation's candidate arcs (each
state keeps its `candidate_arc_id`, so the candidate-to-road-segment correspondence is preserved).

**Emission:** `w · log P(reference | features)` from the final classifier.

**Transition** from arc *i* (observation *t−1*) to arc *j* (observation *t*), using the directed road graph:

| relation | condition | log-score |
|---|---|---|
| same arc | i = j | 0, minus `back_pen` per 10 m moved *backwards* along the arc (beyond 5 m of GPS noise) |
| successor | to(i) = from(j), not a U-turn | `next_pen` |
| U-turn | j is the reverse of i (from(j) = to(i) and to(j) = from(i)) | `uturn_pen` |
| two hops | an arc x with from(x) = to(i), to(x) = from(j) | `two_pen` |
| anything else (invalid / far) | — | `far_pen` (finite, so a path always exists) |

**GPS gaps:** penalties are multiplied by `1 / (1 + max(0, Δt − 1) / tau)` so longer gaps allow
bigger moves; if `Δt > gap_reset` seconds the transition is reset to uniform (no constraint).
Observations without candidates (candidate misses) are skipped; Δt is measured between the
observations that remain.

**Viterbi recurrence:** δ_t(j) = max_i [δ_{t−1}(i) + A(i, j)] + B_t(j), with back-pointers; the
decoded path gives one predicted arc for every observation that has candidates.

**Decoder tuning:** `w, next_pen, uturn_pen, two_pen, far_pen, back_pen, tau, gap_reset` are tuned with Optuna on
**training trips only**, using the classifier's **out-of-fold** probabilities (so the decoder never
sees probabilities from a model trained on the same trip).

In [ ]:
def prepare_trips(df, proba, split):
    """Per trip: list of per-observation candidate arrays plus precomputed relation codes."""
    d = df[POINT + ["candidate_arc_id", TARGET, "pos_m"]].copy(); d["p"] = np.asarray(proba)
    trips = {}
    for tid, g in d.groupby("trip_id", sort=False):
        frm, to, tt = RAW[split][tid]["frm"], RAW[split][tid]["to"], RAW[split][tid]["t"]
        succ_to = defaultdict(set)                      # node -> nodes reachable with one arc
        for f_, t_ in zip(frm, to):
            succ_to[f_].add(t_)
        obs = [(gi, gg.candidate_arc_id.to_numpy(), gg.p.to_numpy(), gg[TARGET].to_numpy(), gg.pos_m.to_numpy())
               for gi, gg in g.groupby("gps_index", sort=True)]
        codes, dts, backs = [None], [0.0], [None]
        for k in range(1, len(obs)):
            a_prev, a_cur = obs[k - 1][1], obs[k][1]
            same = a_prev[:, None] == a_cur[None, :]
            nxt = to[a_prev][:, None] == frm[a_cur][None, :]
            two = np.array([[frm[b] in succ_to[to[a]] for b in a_cur] for a in a_prev])
            uturn = (to[a_prev][:, None] == frm[a_cur][None, :]) & (frm[a_prev][:, None] == to[a_cur][None, :])
            c = np.full(same.shape, 3, dtype=np.int8); c[two] = 2; c[nxt] = 1; c[uturn] = 4; c[same] = 0
            back = np.where(same, np.clip((obs[k - 1][4][:, None] - obs[k][4][None, :] - 5.0) / 10.0, 0, 3), 0.0)
            codes.append(c); backs.append(back); dts.append(float(tt[obs[k][0]] - tt[obs[k - 1][0]]))
        trips[tid] = dict(obs=obs, codes=codes, dts=dts, backs=backs)
    return trips

def viterbi_trip(trip, hp):
    pen = np.array([0.0, hp["next_pen"], hp["two_pen"], hp["far_pen"], hp["uturn_pen"]])
    obs = trip["obs"]
    delta = hp["w"] * np.log(np.clip(obs[0][2], 1e-6, 1)); back = [None]
    for k in range(1, len(obs)):
        dt = trip["dts"][k]
        if dt > hp["gap_reset"]:
            A = np.zeros(trip["codes"][k].shape)
        else:
            A = (pen[trip["codes"][k]] + hp["back_pen"] * trip["backs"][k]) / (1.0 + max(0.0, dt - 1.0) / hp["tau"])
        scores = delta[:, None] + A
        bp = scores.argmax(axis=0)
        delta = scores[bp, np.arange(scores.shape[1])] + hp["w"] * np.log(np.clip(obs[k][2], 1e-6, 1))
        back.append(bp)
    path = [int(delta.argmax())]
    for k in range(len(obs) - 1, 0, -1):
        path.append(int(back[k][path[-1]]))
    path.reverse()
    return [(obs[k][0], int(obs[k][1][s]), int(obs[k][3][s])) for k, s in enumerate(path)]

def decode(trips, hp):
    rows = [(tid, gi, arc, lab) for tid, tr in trips.items() for gi, arc, lab in viterbi_trip(tr, hp)]
    return pd.DataFrame(rows, columns=["trip_id", "gps_index", "candidate_arc_id", TARGET])

DEFAULT_HP = dict(w=1.0, next_pen=-0.5, uturn_pen=-6.0, two_pen=-2.0, far_pen=-8.0, back_pen=-1.0, tau=10.0, gap_reset=60.0)

# --- tune the decoder on training trips with out-of-fold probabilities
rng = np.random.default_rng(SEED)
tune_trips = sorted(rng.choice(sorted(train.trip_id.unique()), size=min(HMM_TUNE_TRIPS, train.trip_id.nunique()), replace=False))
mask = train.trip_id.isin(tune_trips).to_numpy()
t0 = time.perf_counter()
tune_struct = prepare_trips(train[mask], FINAL_OOF[mask], "train")
print(f"Prepared {len(tune_struct)} training trips for decoder tuning in {time.perf_counter()-t0:.1f}s")
print(f"Classifier-only OOF Top-1 on these trips: {top1_choice(train[mask], FINAL_OOF[mask])[TARGET].mean():.4f}")
print(f"HMM with default parameters:          {decode(tune_struct, DEFAULT_HP)[TARGET].mean():.4f}")

def hmm_objective(trial):
    hp = dict(w=trial.suggest_float("w", 0.2, 3.0, log=True),
              next_pen=trial.suggest_float("next_pen", -3.0, 0.0),
              uturn_pen=trial.suggest_float("uturn_pen", -20.0, -0.5),
              two_pen=trial.suggest_float("two_pen", -8.0, -0.2),
              back_pen=trial.suggest_float("back_pen", -6.0, 0.0),
              far_pen=trial.suggest_float("far_pen", -25.0, -2.0),
              tau=trial.suggest_float("tau", 1.0, 120.0, log=True),
              gap_reset=trial.suggest_float("gap_reset", 5.0, 600.0, log=True))
    return decode(tune_struct, hp)[TARGET].mean()

hmm_study = optuna.create_study(direction="maximize", sampler=optuna.samplers.TPESampler(seed=SEED))
hmm_study.enqueue_trial(DEFAULT_HP)
t0 = time.perf_counter()
hmm_study.optimize(hmm_objective, n_trials=HMM_TRIALS,
                   callbacks=[lambda s, t: print(f"  hmm trial {t.number:2d}: top1={t.value:.4f}", flush=True)])
HMM_TUNE_S = time.perf_counter() - t0
HMM_HP = hmm_study.best_params
hmm_study.trials_dataframe().to_csv(os.path.join(OUT_DIR, "hmm_tuning_trials.csv"), index=False)
print(f"\nBest decoder parameters (training trips, OOF): {json.dumps({k: round(v, 4) for k, v in HMM_HP.items()})}")
print(f"Decoder tuning: {HMM_TRIALS} trials in {HMM_TUNE_S/60:.1f} min")
log_event("hmm_tuning", trips=len(tune_trips), trials=HMM_TRIALS, best=hmm_study.best_value,
          params=HMM_HP, seconds=HMM_TUNE_S)

## 11. Classifier-only vs classifier + HMM on the student-test trips

- **Pointwise:** observation-level Top-1 accuracy (and end-to-end accuracy counting candidate misses as wrong).
- **Sequence-level:** per-trip accuracy; route recovery against the `.route` file — the predicted arc
  sequence (consecutive repeats merged) is compared with the reference route by longest common
  subsequence (LCS precision = LCS / predicted length, LCS recall = LCS / reference length).
- **Route continuity** and the number of disconnected jumps.

In [ ]:
def lcs_len(a, b):
    prev = [0] * (len(b) + 1)
    for x in a:
        cur = [0]
        for j, y in enumerate(b):
            cur.append(prev[j] + 1 if x == y else max(prev[j + 1], cur[j]))
        prev = cur
    return prev[-1]

def sequence_report(choice, split, label):
    rows = []
    for tid, g in choice.groupby("trip_id"):
        a = g.sort_values("gps_index")["candidate_arc_id"].to_numpy()
        seq = a[np.r_[True, a[1:] != a[:-1]]]
        ref = RAW[split][tid]["route"]
        frm, to = RAW[split][tid]["frm"], RAW[split][tid]["to"]
        ch = a[1:] != a[:-1]
        jumps = int((to[a[:-1][ch]] != frm[a[1:][ch]]).sum())
        L = lcs_len(list(seq), list(ref))
        rows.append(dict(trip_id=tid, method=label, points=len(g), top1=g[TARGET].mean(),
                         lcs_precision=L / len(seq), lcs_recall=L / len(ref), jumps=jumps,
                         continuity=1 - jumps / max(int(ch.sum()), 1)))
    return pd.DataFrame(rows)

t0 = time.perf_counter()
test_struct = prepare_trips(test, FINAL_TEST_P, "test")
clf_choice = top1_choice(test, FINAL_TEST_P)
hmm_choice = decode(test_struct, HMM_HP)
HMM_TEST_S = time.perf_counter() - t0

per_trip = pd.concat([sequence_report(clf_choice, "test", "classifier only"),
                      sequence_report(hmm_choice, "test", "classifier + HMM")])
per_trip.to_csv(os.path.join(OUT_DIR, "student_test_per_trip.csv"), index=False)

def overall(choice, label):
    return {"method": label, "top1_accuracy": choice[TARGET].mean(),
            "end_to_end_accuracy": choice[TARGET].sum() / sum(RAW["test"][t]["n_gps"] for t in set(test.trip_id)),
            "route_continuity": route_continuity(choice, "test"),
            "mean_trip_top1": per_trip[per_trip.method == label].top1.mean(),
            "worst_trip_top1": per_trip[per_trip.method == label].top1.min(),
            "lcs_precision": per_trip[per_trip.method == label].lcs_precision.mean(),
            "lcs_recall": per_trip[per_trip.method == label].lcs_recall.mean(),
            "disconnected_jumps": int(per_trip[per_trip.method == label].jumps.sum())}

final_table = pd.DataFrame([{"method": "nearest candidate (no ML)", "top1_accuracy": BASELINE_TEST},
                            overall(clf_choice, "classifier only"), overall(hmm_choice, "classifier + HMM")])
display(final_table.round(4))
final_table.to_csv(os.path.join(OUT_DIR, "classifier_vs_hmm_student_test.csv"), index=False)
display(per_trip.pivot(index="trip_id", columns="method", values="top1").round(4))
print(f"HMM preparation + decoding of {test.trip_id.nunique()} trips: {HMM_TEST_S:.1f}s")

best_top1 = final_table.top1_accuracy.max()
print(f"\nTarget >= {TARGET_TOP1:.0%} observation-level Top-1 on student-test: "
      f"{'MET' if best_top1 >= TARGET_TOP1 else 'NOT MET'} (best = {best_top1:.2%})")
log_event("final_evaluation", **{r["method"]: {k: (float(v) if isinstance(v, (int, float, np.floating)) else v)
                                                for k, v in r.items()} for r in final_table.to_dict("records")})

fig, ax = plt.subplots(1, 2, figsize=(14, 4))
pv = per_trip.pivot(index="trip_id", columns="method", values="top1")
pv.plot.bar(ax=ax[0], color=["#55A868", "#4C72B0"]); ax[0].set_title("Student-test Top-1 per trip")
ax[0].set_ylim(max(0, pv.min().min() - 0.05), 1); ax[0].axhline(TARGET_TOP1, ls="--", c="k", lw=1)
pj = per_trip.pivot(index="trip_id", columns="method", values="jumps")
pj.plot.bar(ax=ax[1], color=["#55A868", "#4C72B0"]); ax[1].set_title("Disconnected jumps per trip (lower is better)")
plt.tight_layout(); plt.savefig(os.path.join(OUT_DIR, "classifier_vs_hmm.png"), dpi=130); plt.show()

## 11b. Distance-aware HMM (Newson & Krumm transition model)

The HMM in section 10 only knows whether two arcs are the same, consecutive, two hops apart or far.
Most remaining errors are at **junction boundaries** (the neighbouring arc of the same road), so this
decoder uses the standard map-matching transition model:

- for consecutive observations, the **network distance** from candidate *i*'s position to candidate
  *j*'s position (driving along the directed road graph; shortest paths up to 750 m) is compared with
  the **straight-line GPS distance** between the two observations;
- transition log-score = −|network distance − GPS distance| / β; moving backwards on the same arc
  or no path within the radius gets `far_pen`; U-turns add `uturn_pen`;
- gaps (Δt > `gap_reset` or a GPS jump > 250 m) reset the transition (no constraint).

Parameters are tuned on the same training trips with out-of-fold probabilities; the student-test
trips are decoded once. Nothing is retrained.

In [ ]:
# ===== 11b. Distance-aware HMM (Newson & Krumm transition model) =====
# Runs after section 11 and reuses what is already in memory: train, test, RAW, FINAL_OOF, FINAL_TEST_P,
# tune_trips, mask, top1_choice, route_continuity, sequence_report. Nothing is retrained.
import networkx as nx

GAP_DIST_M = 250.0      # GPS displacement above which consecutive observations are treated as a gap
SP_CUTOFF_M = 750.0     # shortest-path search radius on the road network

def build_trip_graph(r):
    L = np.hypot(*(r["N"][r["to"]] - r["N"][r["frm"]]).T)
    G = nx.DiGraph()
    G.add_weighted_edges_from(zip(r["frm"].tolist(), r["to"].tolist(), L.tolist()))
    return G, L

def prepare_trips_dist(df, proba, split):
    """Per trip: candidates per observation + network distances between consecutive candidates."""
    d = df[POINT + ["candidate_arc_id", TARGET, "pos_m"]].copy(); d["p"] = np.asarray(proba)
    trips = {}
    for tid, g in d.groupby("trip_id", sort=False):
        r = RAW[split][tid]; G, L = build_trip_graph(r); cache = {}
        def sp_from(node):
            if node not in cache:
                cache[node] = nx.single_source_dijkstra_path_length(G, node, cutoff=SP_CUTOFF_M)
            return cache[node]
        obs = []
        for gi, gg in g.groupby("gps_index", sort=True):
            a = gg.candidate_arc_id.to_numpy()
            obs.append((gi, a, gg.p.to_numpy(), gg[TARGET].to_numpy(), np.clip(gg.pos_m.to_numpy(), 0, L[a])))
        route_d, uturn, gps_d, dts = [None], [None], [0.0], [0.0]
        for k in range(1, len(obs)):
            (g0, a0, _, _, s0), (g1, a1, _, _, s1) = obs[k - 1], obs[k]
            D = np.full((len(a0), len(a1)), np.inf)
            for i, (ai, si) in enumerate(zip(a0, s0)):
                reach = sp_from(int(r["to"][ai]))
                for j, (aj, sj) in enumerate(zip(a1, s1)):
                    if ai == aj:
                        D[i, j] = max(sj - si, 0.0) if sj >= si - 10.0 else np.inf   # no driving backwards
                    else:
                        sp = reach.get(int(r["frm"][aj]))
                        if sp is not None:
                            D[i, j] = (L[ai] - si) + sp + sj
            route_d.append(D)
            uturn.append((r["to"][a0][:, None] == r["frm"][a1][None, :]) & (r["frm"][a0][:, None] == r["to"][a1][None, :]))
            gps_d.append(float(np.hypot(*(r["P"][g1] - r["P"][g0]))))
            dts.append(float(r["t"][g1] - r["t"][g0]))
        trips[tid] = dict(obs=obs, route_d=route_d, uturn=uturn, gps_d=gps_d, dts=dts)
    return trips

def viterbi_dist(trip, hp):
    obs = trip["obs"]
    delta = hp["w"] * np.log(np.clip(obs[0][2], 1e-6, 1)); back = [None]
    for k in range(1, len(obs)):
        if trip["dts"][k] > hp["gap_reset"] or trip["gps_d"][k] > GAP_DIST_M:
            A = np.zeros(trip["route_d"][k].shape)                       # GPS gap: unconstrained
        else:
            D = trip["route_d"][k]
            A = np.where(np.isfinite(D), -np.abs(np.nan_to_num(D, posinf=0.0) - trip["gps_d"][k]) / hp["beta"], hp["far_pen"])
            A = A + hp["uturn_pen"] * trip["uturn"][k]
        scores = delta[:, None] + A
        bp = scores.argmax(axis=0)
        delta = scores[bp, np.arange(scores.shape[1])] + hp["w"] * np.log(np.clip(obs[k][2], 1e-6, 1))
        back.append(bp)
    path = [int(delta.argmax())]
    for k in range(len(obs) - 1, 0, -1):
        path.append(int(back[k][path[-1]]))
    path.reverse()
    return [(obs[k][0], int(obs[k][1][s]), int(obs[k][3][s])) for k, s in enumerate(path)]

def decode_dist(trips, hp):
    rows = [(tid, gi, arc, lab) for tid, tr in trips.items() for gi, arc, lab in viterbi_dist(tr, hp)]
    return pd.DataFrame(rows, columns=["trip_id", "gps_index", "candidate_arc_id", TARGET])

# --- tune on training trips (out-of-fold probabilities) only
t0 = time.perf_counter()
tune_struct2 = prepare_trips_dist(train[mask], FINAL_OOF[mask], "train")
print(f"Prepared {len(tune_struct2)} training trips (network distances) in {time.perf_counter()-t0:.0f}s", flush=True)
HP2_DEFAULT = dict(w=1.0, beta=20.0, uturn_pen=-5.0, far_pen=-15.0, gap_reset=60.0)
print(f"Distance-aware HMM, default parameters, training trips: {decode_dist(tune_struct2, HP2_DEFAULT)[TARGET].mean():.4f}")

def hmm2_objective(trial):
    hp = dict(w=trial.suggest_float("w", 0.2, 3.0, log=True),
              beta=trial.suggest_float("beta", 2.0, 200.0, log=True),      # tolerance on |route - GPS| distance (m)
              uturn_pen=trial.suggest_float("uturn_pen", -20.0, 0.0),
              far_pen=trial.suggest_float("far_pen", -40.0, -2.0),
              gap_reset=trial.suggest_float("gap_reset", 5.0, 300.0, log=True))
    return decode_dist(tune_struct2, hp)[TARGET].mean()

hmm2_study = optuna.create_study(direction="maximize", sampler=optuna.samplers.TPESampler(seed=SEED))
hmm2_study.enqueue_trial(HP2_DEFAULT)
t0 = time.perf_counter()
hmm2_study.optimize(hmm2_objective, n_trials=max(HMM_TRIALS, 20),
                    callbacks=[lambda s, t: print(f"  trial {t.number:2d}: top1={t.value:.4f}", flush=True)])
HMM2_HP = hmm2_study.best_params
HMM2_TUNE_S = time.perf_counter() - t0
hmm2_study.trials_dataframe().to_csv(os.path.join(OUT_DIR, "hmm_distance_tuning_trials.csv"), index=False)
print(f"Best (training trips, OOF): {hmm2_study.best_value:.4f} with {json.dumps({k: round(v, 3) for k, v in HMM2_HP.items()})}")

# --- evaluate once on the student-test trips
t0 = time.perf_counter()
test_struct2 = prepare_trips_dist(test, FINAL_TEST_P, "test")
hmm2_choice = decode_dist(test_struct2, HMM2_HP)
HMM2_TEST_S = time.perf_counter() - t0
per_trip2 = pd.concat([per_trip, sequence_report(hmm2_choice, "test", "classifier + distance HMM")])
per_trip2.to_csv(os.path.join(OUT_DIR, "student_test_per_trip_all_methods.csv"), index=False)

def overall2(choice, label, pt):
    s = pt[pt.method == label]
    return {"method": label, "top1_accuracy": choice[TARGET].mean(),
            "end_to_end_accuracy": choice[TARGET].sum() / sum(RAW["test"][t]["n_gps"] for t in set(test.trip_id)),
            "route_continuity": route_continuity(choice, "test"), "mean_trip_top1": s.top1.mean(),
            "worst_trip_top1": s.top1.min(), "lcs_precision": s.lcs_precision.mean(),
            "lcs_recall": s.lcs_recall.mean(), "disconnected_jumps": int(s.jumps.sum())}

final_table2 = pd.DataFrame([{"method": "nearest candidate (no ML)", "top1_accuracy": BASELINE_TEST},
                             overall2(clf_choice, "classifier only", per_trip2),
                             overall2(hmm_choice, "classifier + HMM", per_trip2),
                             overall2(hmm2_choice, "classifier + distance HMM", per_trip2)])
display(final_table2.round(4))
final_table2.to_csv(os.path.join(OUT_DIR, "classifier_vs_hmm_vs_distance_hmm.csv"), index=False)
display(per_trip2.pivot(index="trip_id", columns="method", values="top1").round(4))
print(f"Distance HMM on {test.trip_id.nunique()} test trips: {HMM2_TEST_S:.1f}s; tuning {HMM2_TUNE_S/60:.1f} min")
best2 = final_table2.top1_accuracy.max()
print(f"\nTarget >= {TARGET_TOP1:.0%} observation-level Top-1 on student-test: "
      f"{'MET' if best2 >= TARGET_TOP1 else 'NOT MET'} (best = {best2:.2%})")
with open(os.path.join(OUT_DIR, "hmm_distance_config.json"), "w") as fh:
    json.dump({"params": HMM2_HP, "gap_dist_m": GAP_DIST_M, "sp_cutoff_m": SP_CUTOFF_M,
               "transition": "-|network distance - GPS distance| / beta; unreachable within cutoff or backwards on the same arc = far_pen; U-turn adds uturn_pen; reset if dt>gap_reset or GPS jump>gap_dist_m",
               "emission": "w * log(p)", "tuned_on_training_trips": [str(t) for t in tune_trips]}, fh, indent=2)
joblib.dump({"hmm_distance_params": HMM2_HP, "gap_dist_m": GAP_DIST_M, "sp_cutoff_m": SP_CUTOFF_M},
            os.path.join(OUT_DIR, "hmm_distance_config.joblib"))

## 12. Save artifacts and download

Everything is written to `pathsnap_results/` and downloaded as `pathsnap_results.zip`
(Colab deletes uploaded and generated files when the runtime resets).

In [ ]:
config = {
    "seed": SEED, "profile": PROFILE, "settings": {k: (v if not isinstance(v, dict) else v) for k, v in CFG.items()},
    "files": {k: os.path.basename(v) for k, v in paths.items()},
    "features": FEATURES, "knn_features": KNN_FEATURES,
    "models": {n: {"family": s["family"], "config": s["config"]} for n, s in MODELS.items()},
    "selection_rule": "highest mean CV Top-1 on training trips; ties within 1 std broken by lower CV log loss",
    "selected_model": SELECTED, "tuned": bool(USE_TUNED),
    "final_params": {k: str(v) for k, v in FINAL_PARAMS.items()},
    "hmm": {"params": HMM_HP, "tuned_on_training_trips": [str(t) for t in tune_trips],
            "transition": "same=0 (+back_pen per 10 m backwards), successor=next_pen, U-turn=uturn_pen, two-hop=two_pen, other=far_pen; scaled by 1/(1+max(0,dt-1)/tau); reset if dt>gap_reset",
            "emission": "w * log(p)"},
    "candidate_miss_rate": MISS_RATE,
    "versions": {"python": platform.python_version(), "sklearn": sklearn.__version__,
                 "lightgbm": lightgbm.__version__, "optuna": optuna.__version__},
}
with open(os.path.join(OUT_DIR, "config.json"), "w") as fh:
    json.dump(config, fh, indent=2, default=str)
with open(os.path.join(OUT_DIR, "experiment_log.json"), "w") as fh:
    json.dump(EXPERIMENT_LOG, fh, indent=2, default=str)
joblib.dump({"hmm_params": HMM_HP, "features": FEATURES}, os.path.join(OUT_DIR, "hmm_config.joblib"))
out = test[META + [TARGET]].assign(probability=FINAL_TEST_P)
out.to_csv(os.path.join(OUT_DIR, "student_test_candidate_probabilities.csv"), index=False)
hmm_choice.rename(columns={"candidate_arc_id": "hmm_arc"}).merge(
    clf_choice.rename(columns={"candidate_arc_id": "classifier_arc"})[["trip_id", "gps_index", "classifier_arc"]],
    on=["trip_id", "gps_index"]).to_csv(os.path.join(OUT_DIR, "student_test_predicted_route.csv"), index=False)

zip_path = os.path.join(DATA_DIR, "pathsnap_results.zip")
with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as z:
    for f in sorted(os.listdir(OUT_DIR)):
        z.write(os.path.join(OUT_DIR, f), f)
print("Saved:", *sorted(os.listdir(OUT_DIR)), sep="\n  ")
if IN_COLAB:
    colab_files.download(zip_path)

## 13. Observations (write from the numbers above)

- Which model family performed best on CV Top-1, and was it stable across folds?
- Did Bayesian optimisation help on CV, and on the student-test trips?
- How much did the HMM change Top-1, route continuity and disconnected jumps? On which trips did it help or hurt?
- How large is the candidate miss rate, and what does it imply for end-to-end accuracy?
- Why is the final model compatible with the sequential decoder (probability quality, speed)?